In [1]:
from data_preparation import prepare_splink_data


from pathlib import Path
import os
from dotenv import load_dotenv
import pandas as pd


#################
# Configuración #
#################
load_dotenv()
DATA_DIR = os.getenv("DATA_DIR", "data")
DATASETS_PATH = Path(DATA_DIR) / "cleaned" / "datasets"

# Data Prep

In [2]:
df_A_original = prepare_splink_data(
    parquet_path= DATASETS_PATH / "dataset_A.parquet",
    date_columns=[
        "fecha_nacim"
    ],
    sex_columns=[
        "sexo"
    ]
)

# Keep ground truth separately
df_A = df_A_original.copy()
df_truth = df_A[
    ["unique_id", "id_persona"]
].copy()
# id_registro empieza en 0 unique_id en 1
df_A = df_A.drop(columns=["id_registro", "id_persona"]) # ground truth
df_A.head()

,unique_id,sexo,cve_lengua_nom,nacesp_cve_nom,fecha_nacim,nombre,apellido_paterno,apellido_materno,ent_resid_nom,tloc_resid_nom,tipo_defun_nom,par_agre_nom,vio_fami_nom,cirugia_nom,natviole_nom
0,1,masculino,<NA>,mexicana no especificado de entidad federativa,1954-01-22,jose,martinez,castaneda,sinaloa,250 000 a 499 999 habitantes,enfermedad,no hubo violencia,no aplica cuando no es homicidio,no especificado a,no
1,2,femenino,no aplica,mexicana no especificado de entidad federativa,1931-02-09,maria antonia,cerda,ramirez,san luis potosi,1 000 a 1 999 habitantes,defuncion por enfermedad,no hubo violencia,no aplica cuando no es homicidio,no especificada,no
2,3,femenino,no aplica,mexicana no especificado de entidad federativa,1960-04-20,ma isabel,carbajal,de jesus,mexico,250 000 a 499 999 habitantes,enfermedad muerte natural,no hubo violencia,no aplica cuando no es homicidio,no especificada,no
3,4,masculino,no aplica,mexicana no especificado de entidad federativa,1952-06-15,jorge luis,hernandez,limon,tamaulipas,2 500 a 4 999 habitantes,enfermedad muerte natural,no huob violencia,no aplica cuando no es homicidio,no especificada,no
4,5,femenino,no aplica,mexicana no especificado de entidad federativa,1978-11-23,juana,lira,rodriguez,chihuahua mexico,1 500 000 y mas habitantes,enfermedad muerte natural,no hubo violencia,no aplica cuando no es homicidio,si,0


In [3]:
df_A.size

1419750

# Data Missingness DUCK DB

In [4]:
from splink.exploratory import completeness_chart
from splink import DuckDBAPI
db_api = DuckDBAPI()
db_api._con.execute("SET threads = 4")
db_api._con.execute("SET preserve_insertion_order = false")
db_api._con.execute("SET memory_limit = '8GB'")
df_sdf = db_api.register(df_A)
display(completeness_chart(df_sdf))

SplinkChart (use .to_html() or .save() to export HTML)

# Data distribution

In [5]:
from splink.exploratory import profile_columns

profile_columns(df_sdf, top_n=10, bottom_n=5)

SplinkChart (use .to_html() or .save() to export HTML)

**Propósito**: 
Encontrar typos por ejemplo 
- sexo se puede estandarizar como se hace en las fechas (✅)
- quitar acentos hay "sí" y "si" ✅
- estandarizar entidades ✅
Identificar Variables débiles
- cve_lengua_nom
- nacesp_cve_nom

In [6]:
from data_preparation import normalize_entities

df_A, report = normalize_entities(
    df_A,
    column="ent_resid_nom",
    max_distance=3,
)

display(report)

,unique_id,original,normalized,distance
0,2,san luis potosi,san luis potosí,0
1,3,mexico,estado de méxico,0
2,10,michoacan de ocampo,michoacán,0
3,12,mexico,estado de méxico,0
4,13,mexico,estado de méxico,0
...,...,...,...,...
40714,94640,veracruz de ignacio de la llave,veracruz,0
40715,94642,nuevo leon,nuevo león,0
40716,94643,mexico,estado de méxico,0
40717,94645,michoacan de ocampo,michoacán,0


In [7]:
report["distance"].value_counts()

distance
0    40719
Name: count, dtype: int64

In [8]:
from splink import DuckDBAPI
db_api = DuckDBAPI()
db_api._con.execute("SET threads = 4")
db_api._con.execute("SET preserve_insertion_order = false")
db_api._con.execute("SET memory_limit = '8GB'")
df_sdf = db_api.register(df_A)

# Blocking Rules

In [8]:
from splink import block_on
from splink.blocking_analysis import (
    chart_comparisons_from_blocking_rules,
)

# blocking_rules = [ # pueden ser más complejas o agregar más reglas 
#     block_on("sexo"),
#     block_on("ent_resid_nom"),
#     # block_on("sexo", "ent_resid_nom") es redundante
# ]

blocking_rules = [
    block_on("fecha_nacim", "sexo"),
    block_on("apellido_paterno", "fecha_nacim"),
    block_on("apellido_paterno", "apellido_materno"),
]

blocking_rules = [
    # originales
    block_on("fecha_nacim", "sexo"),
    block_on("apellido_paterno", "fecha_nacim"),
    block_on("apellido_paterno", "apellido_materno"),
    # nuevas: cada una omite un campo distinto
    block_on("nombre", "apellido_paterno"),
    block_on("nombre", "fecha_nacim"),
    block_on("apellido_materno", "fecha_nacim"),
    # parcial: prefijos + año de nacimiento
    block_on("substr(nombre,1,3)", "substr(apellido_paterno,1,3)", "substr(fecha_nacim,1,4)"),
]

chart_comparisons_from_blocking_rules(
    df_sdf,
    blocking_rules=blocking_rules,
    link_type="dedupe_only",
    record_sample_proportion=0.3,
)
          

/Users/daragama/Documents/Master25/Tesis/multi-cardinal-record-linkage/.venv/lib/python3.14/site-packages/splink/internals/blocking_analysis.py:668: UserWarning: The sampled blocking analysis estimate for blocking rule '(l."apellido_paterno" = r."apellido_paterno") AND (l."fecha_nacim" = r."fecha_nacim")' is based on 126 sampled pairwise comparisons. This is below the recommended minimum of 1,000, so the estimate may be unstable. Increase record_sample_proportion for a more stable estimate.
  return _cumulative_comparisons_to_be_scored_from_blocking_rules(
/Users/daragama/Documents/Master25/Tesis/multi-cardinal-record-linkage/.venv/lib/python3.14/site-packages/splink/internals/blocking_analysis.py:668: UserWarning: The sampled blocking analysis estimate for blocking rule '(l."nombre" = r."nombre") AND (l."fecha_nacim" = r."fecha_nacim")' is based on 21 sampled pairwise comparisons. This is below the recommended minimum of 1,000, so the estimate may be unstable. Increase record_sample_p

SplinkChart (use .to_html() or .save() to export HTML)

# Model Settings

In [10]:
print(df_A.columns)

Index(['unique_id', 'sexo', 'cve_lengua_nom', 'nacesp_cve_nom', 'fecha_nacim',
       'nombre', 'apellido_paterno', 'apellido_materno', 'ent_resid_nom',
       'tloc_resid_nom', 'tipo_defun_nom', 'par_agre_nom', 'vio_fami_nom',
       'cirugia_nom', 'natviole_nom'],
      dtype='str')


Dejamos fuera:
- "tipo_defun_nom", cl.ExactMatch("tipo_defun_nom"),
- "par_agre_nom", cl.ExactMatch("tipo_defun_nom"),
- "vio_fami_nom",
- "cirugia_nom",
- "natviole_nom",

In [11]:
import splink.comparison_library as cl

comparisons_dic = {
    "sexo": cl.ExactMatch("sexo"),
    "cve_lengua_nom": cl.ExactMatch("cve_lengua_nom"),
    "nacesp_cve_nom": cl.ExactMatch("nacesp_cve_nom"),
    "fecha_nacim": cl.DateOfBirthComparison(
        "fecha_nacim",
        input_is_string=True,
        datetime_format="%Y-%m-%d",
        datetime_thresholds=[1, 1, 5], # se pueden optimizar
        datetime_metrics=["day", "month", "year"],
    ),
    "nombre": cl.NameComparison("nombre"),
    "apellido_paterno": cl.NameComparison("apellido_paterno"),
    "apellido_materno": cl.NameComparison("apellido_materno"),
    "ent_resid_nom": cl.ExactMatch("ent_resid_nom").configure(term_frequency_adjustments=True),
    "tloc_resid_nom": cl.JaroWinklerAtThresholds(
        "tloc_resid_nom",
        [0.95, 0.90, 0.80],
    ),
}

In [12]:
from splink import Linker, SettingsCreator, block_on, DuckDBAPI, ColumnExpression

comparisons_m2 = [
    comparisons_dic["sexo"],
    comparisons_dic["fecha_nacim"],
    comparisons_dic["nombre"],
    comparisons_dic["apellido_paterno"],
    comparisons_dic["apellido_materno"],
    comparisons_dic["ent_resid_nom"],
    comparisons_dic["tloc_resid_nom"]
]

settings = SettingsCreator(
    link_type="dedupe_only", # por ahora
    comparisons=comparisons_m2,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)

linker = Linker(df_sdf, settings)

## Estimate $u$ and $m$

- $u$ values are the proportion of records falling into each ComparisonLevel amongst truly non-matching records.
- $m$ values are the proportion of records falling into each ComparisonLevel amongst truly matching records.

`probability_two_random_records_match` probabilidad de que dos registros seleccionados aleatoriamente sean un match. En caso bipartito se conoce, aquí se debe estimar
Reglas muy estríctas, deben encontrar un subconjunto de parejas que tengan una probabilidad muy alta de ser coincidencias verdaderas. suficientemente discriminantes para identificar por sí solas a una persona

In [13]:
# Se puede ajustar 
SUPPOSED_RECALL = 0.7 # porcentaje de todos los matches verdaderos que se recuperan con las reglas determinísticas. 



deterministic_rules = [
    # Nombre completo y fecha de nacimiento exactos
    """
    l.nombre = r.nombre
    AND l.apellido_paterno = r.apellido_paterno
    AND l.apellido_materno = r.apellido_materno
    AND l.fecha_nacim = r.fecha_nacim
    """,

    # Ambos apellidos y fecha de nacimiento exactos
    """
    l.apellido_paterno = r.apellido_paterno
    AND l.apellido_materno = r.apellido_materno
    AND l.fecha_nacim = r.fecha_nacim
    """,

    # Nombre y apellido paterno exactos, fecha con tolerancia de un día
    """
    l.nombre = r.nombre
    AND l.apellido_paterno = r.apellido_paterno
    AND ABS(
        DATE_DIFF('day', 
                    CAST(l.fecha_nacim AS DATE), 
                    CAST(r.fecha_nacim AS DATE)
        )
        ) <= 1
    """,
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=SUPPOSED_RECALL)


Probability two random records match is estimated to be  1.42e-07.
This means that amongst all possible pairwise record comparisons, one in 7,050,387.48 are expected to match.  With 265,960,759,785 total possible comparisons, we expect a total of around 37,722.86 matching pairs


In [14]:

# 1. Crear linker desde cero

linker = Linker(df_sdf, settings)

# 2. Estimar u mediante muestreo aleatorio
linker.training.estimate_u_using_random_sampling(
    max_pairs=1e7,
    seed=42
)

# 3. Estimar m bloqueando por fecha
training_session_1 = (
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on("fecha_nacim")
    )
)

# 4. Estimar m bloqueando por nombres y apellidos
training_session_2 = (
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on(
            "nombre",
            "apellido_paterno",
            "apellido_materno"
        )
    )
)

----- Estimating u probabilities using random sampling -----
Estimating u with: max_pairs = 10,000,000, min_count_per_level = 100, num_chunks = 10

Estimating u for: sexo (Comparison 1 of 7)
  Running probe chunk (~1.00% of max_pairs)
  Min u_count: 54,757 for comparison level All other comparisons (cvv=0)
  Exiting early since min count of 54,757 exceeds min_count_per_level = 100

Estimating u for: fecha_nacim (Comparison 2 of 7)
  Running probe chunk (~1.00% of max_pairs)
  Min u_count: 0 for comparison level Abs date difference <= 1 day (cvv=3)
  Probe did not converge; restarting with normal chunking

  Running chunk 1/10
  Count of 7 for level Abs date difference <= 1 day (cvv=3). Chunk took 0.5 seconds.
  Min u_count not hit, continuing.
  Running chunk 2/10
  Count of 21 for level Abs date difference <= 1 day (cvv=3). Chunk took 0.5 seconds.
  Min u_count not hit, continuing.
  Running chunk 3/10
  Count of 32 for level Abs date difference <= 1 day (cvv=3). Chunk took 0.5 second

### $m$ y $u$ charts

In [15]:
linker.visualisations.m_u_parameters_chart()

SplinkChart (use .to_html() or .save() to export HTML)

In [ ]:
# linker.visualisations.parameter_estimate_comparisons_chart()

: 

## Evaluación

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# Generate predictions
df_predictions = linker.inference.predict()

predictions = df_predictions.as_pandas_dataframe()

# Select a probability threshold
threshold = 0.5

predictions = predictions[
    predictions["match_probability"] >= threshold
].copy()

# Map ground truth
truth_map = (
    df_truth
    .assign(unique_id=lambda df: df["id_registro"] + 1)
    .set_index("unique_id")["id_persona"]
)

# Cada registro debe tener un unique_id diferente
assert truth_map.index.is_unique

# Identificadores presentes en las predicciones
predicted_ids = pd.concat([
    predictions["unique_id_l"],
    predictions["unique_id_r"]
]).unique()

# Comprobar que todos existan en el ground truth
missing_ids = set(predicted_ids) - set(truth_map.index)

print("IDs sin correspondencia:", len(missing_ids))

predictions["id_persona_l"] = (
    predictions["unique_id_l"].map(truth_map)
)

predictions["id_persona_r"] = (
    predictions["unique_id_r"].map(truth_map)
)

# True labels
valid = (
    predictions["id_persona_l"].notna()
    & predictions["id_persona_r"].notna()
)

predictions = predictions.loc[valid].copy()

y_true = (
    predictions["id_persona_l"]
    == predictions["id_persona_r"]
).astype(int)

# Predicted labels: all selected pairs are matches
y_pred = np.ones(len(predictions), dtype=int)

# Metrics
precision = precision_score(y_true, y_pred, zero_division=0)

Blocking time: 1.84 seconds
Predict time (post-blocking): 28.43 seconds


In [ ]:
# Total number of true matching pairs
group_sizes = (
    df_truth.dropna(subset=["id_persona"])
    .groupby("id_persona")
    .size()
)

total_true_pairs = (
    group_sizes * (group_sizes - 1) // 2
).sum()

# True positives
tp = int(y_true.sum())

# False positives
fp = len(predictions) - tp

# False negatives
fn = int(total_true_pairs - tp)

# Metrics
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

results = pd.DataFrame([{
    "threshold": 0.5,
    "TP": tp,
    "FP": fp,
    "FN": fn,
    "precision": precision,
    "recall": recall,
    "f1": f1,
}])

results

,threshold,TP,FP,FN,precision,recall,f1
0,0.5,0,10,34523,0.0,0.0,0
